In [ ]:
# Copyright 2025 DeepMind Technologies Limited. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

[![Colab で開く](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-gemini/genai-processors/blob/main/notebooks/processor_intro.ipynb)



# はじめに

GenAI Processors ライブラリの使い方を段階的に学ぶチュートリアルです。


## 1. 🛠️ セットアップ

まず、GenAI Processors ライブラリをインストールします:


In [ ]:
!pip install genai-processors

### API キー

GenAI のモデルプロセッサを使用するには API キーが必要です。まだお持ちでない場合は Google AI Studio から取得し、Colab のシークレットとしてインポート（推奨）するか、以下で直接設定してください。API キーがなくても本チュートリアルは実行できますが、「GenAI モデルをプロセッサとして使用」のセクションはスキップする必要があります。


In [ ]:
from google.colab import userdata

API_KEY = userdata.get('GOOGLE_API_KEY')

## 2. 💡 基本概念の理解

GenAI Processors ライブラリは主に次の3つの概念を中心に構成されています:

*   **`ProcessorPart`:** プロセッサが使用する標準データオブジェクトです。テキスト、画像、構造化データなど、特定のモダリティの単一のコンテンツを表します。各 `ProcessorPart` には `mimetype` や `substream_name` といったメタデータを付与でき、コンテンツの分類やルーティングに利用します。
*   **`Processor`:** 非同期ストリーム（AsyncIterable）の `ProcessorPart` を入力として受け取り、非同期ストリームの `ProcessorPart` を出力する処理単位です。プロセッサ同士を連結して複雑なパイプラインを構築できます。
*   **`PartProcessor`:** ストリーム内の各パートを独立して処理できる場合のための特化した Processor です。`PartProcessor` は単一の `ProcessorPart` を受け取り、非同期ストリームの `ProcessorPart` を返します。ライブラリは、入力ストリーム内の各 `ProcessorPart` に対して PartProcessor を並行実行し、出力を正しい順序で組み立てることを担います。これにより、複数の PartProcessor を連続で使用する場合でも、独立したパートの効率的な並行処理が可能になります。

**注意**: `ProcessorPart` と `PartProcessor` は名称が似ているため混同しがちですが、指すものは異なります:

> *   `ProcessorPart` は単一のコンテンツを表す **データオブジェクト** です。
> *   `PartProcessor` は個々の `ProcessorPart` を処理するために設計された **Processor** です。


## 3. 🔨 シンプルな Processor の作成

任意の `.` 文字を `EoS` タグに置き換えるシンプルなプロセッサを作成してみましょう。以下では、非同期ジェネレーター関数を `Processor` オブジェクトに変換する `@processor.processor_function` デコレータを使用しています。これは単一の関数から Processor を作るための便利な方法です。


In [ ]:
from collections.abc import AsyncIterable
from genai_processors import content_api, processor


@processor.processor_function
async def simple_text_processor(
    content: AsyncIterable[content_api.ProcessorPart],
) -> AsyncIterable[content_api.ProcessorPart]:
  """Replaces dots with '[EoS]'."""
  async for part in content:
    if content_api.is_text(part.mimetype):
      yield content_api.ProcessorPart(part.text.replace(".", "[EoS]"))
    else:
      yield part

Processor は `processor.Processor` クラスを継承し、`call(..)` メソッドを実装する形でも定義できます。プロセッサが永続的な状態を必要とする場合や、パラメータ化が必要な場合はこの方法を推奨します。


In [ ]:
from genai_processors.core import preamble


class SimpleTextProcessor(processor.Processor):

  def __init__(self, eos_string: str):
    self._eos = eos_string
    # Preamble adds a prefix to a content stream.
    self._preamble = preamble.Preamble("Starting. ")

  async def call(
      self,
      content: AsyncIterable[content_api.ProcessorPart],
  ) -> AsyncIterable[content_api.ProcessorPart]:
    """Replaces dots with '[EoS]'."""
    async for part in self._preamble(content):
      if content_api.is_text(part.mimetype):
        yield content_api.ProcessorPart(part.text.replace(".", self._eos))
      else:
        yield part

## 4. ▶️ Processor の適用

Processor は、`async for` で直接イテレートすることで入力ストリームに適用できます。これは推奨される方法です。

このノートブックでは、`streams.stream_content` ユーティリティをよく使って、`ProcessorPartTypes`（`ProcessorPart` に加えて `str` や `bytes` のような一般的な型も含む）のリストから入力ストリームを作成します。より高度なセットアップでの入力ストリームの作り方は、後述の「Streams と AsyncIterable の扱い」セクションで説明します。

### 非同期での適用［推奨］


In [ ]:
import asyncio
from genai_processors import streams

input_parts = ["Hello", "World"]
input_stream = streams.stream_content(input_parts)

print("\nAsynchronous Output:")
async for part in simple_text_processor(input_stream):
  print(part.text)

`simple_text_processor` は Processor インスタンスである点に注意してください。クラスとして定義した場合は、使用前にインスタンス化が必要です。そのため、次のコードは:

```python
async for part in simple_text_processor(stream):
  ...
```

は次のように置き換える必要があります:

```python
p = SimpleTextProcessor("[EoS]")
async for part in p(stream):
  ...
```


### 同期での適用

同期実行では、`processor.apply_sync` を使って `ProcessorPart` のリストにプロセッサを適用できます。


In [ ]:
import nest_asyncio

nest_asyncio.apply()  # Needed to run async loops in Colab

processed_parts_sync = processor.apply_sync(simple_text_processor, input_parts)

print("Synchronous Output:")
for part in processed_parts_sync:
  print(part.text)

## 5. ⛓️ Processor の連結

このライブラリの真価は、`+` 演算子でプロセッサを連結できる点にあります。


In [ ]:
@processor.processor_function
async def another_text_processor(
    content: AsyncIterable[content_api.ProcessorPart],
) -> AsyncIterable[content_api.ProcessorPart]:
  """Lowercases everything."""
  async for part in content:
    if content_api.is_text(part.mimetype):
      yield content_api.ProcessorPart(part.text.lower())
    else:
      yield part


chained_processor = simple_text_processor + another_text_processor
input_streams = streams.stream_content(["First. Second."])

print("\nChained Processor Output:")
async for part in chained_processor(input_streams):
  print(part.text)

出力は `first[eos] second[eos]` になります。

`+` 演算子は `Processor` と `PartProcessor` の組み合わせを正しく扱います。ただし、可能な限り `PartProcessor` をまとめて配置すると効率が最大化します。

連結時、特別な `debug` および `status` サブストリーム内の `ProcessorPart` は挙動が異なります。これらのサブストリーム内のパートは生成され次第すぐに呼び出し元へ返され、次の Processor には渡されません。


In [ ]:
@processor.processor_function
async def simple_text_processor_with_status(
    content: AsyncIterable[content_api.ProcessorPart],
) -> AsyncIterable[content_api.ProcessorPart]:
  """Replaces dots with '[EoS]'."""
  async for part in content:
    if content_api.is_text(part.mimetype):
      yield content_api.ProcessorPart(part.text.replace(".", "[EoS]"))
      yield processor.status(f"Simple processor done on {part.text}")
    else:
      yield part


chained_processor = simple_text_processor_with_status + another_text_processor
input_streams = streams.stream_content(["First.", "Second."])

print("\nChained Processor Output:")
async for part in chained_processor(input_streams):
  print(part)

`status` サブストリーム内の `ProcessorPart` は、すべて小文字に変換する `another_text_processor` では処理されない点に注意してください。

## 6. 🛣️ Processor の並列実行とスイッチ

プロセッサは、引数としてプロセッサのシーケンスを受け取る `parallel_concat()` 関数を使って並列に実行できます。


In [ ]:
input_stream = streams.stream_content(["First.", "Second."])

p = [another_text_processor, simple_text_processor_with_status]

p = processor.parallel_concat(p)

print("\nParallel Processor Output:")
async for part in p(input_stream):
  print(part)

（`status` パートを除く）出力は `first., second., First[EoS], Second[EoS]` となります。これは `parallel_concat` に渡したプロセッサ一覧の順序に従い、まず `another_text_processor` の出力、その後に `simple_text_processor_with_status` の出力が続きます。

**警告**: どのプロセッサでも処理されずに通過したパートは、複数のプロセッサを通ると出力で繰り返されます。

出力を連結ではなくインターリーブ（交互に混在）したい場合は、「PartProcessor の並列実行」セクションで説明する `PartProcessor` 版の手法を検討してください。

このような並列動作は排他的ではありません。1つのパートが同時に複数のプロセッサで並列処理されます。プロセッサが入力ストリームの異なるパートを排他的に処理する必要がある場合は、スイッチ演算子に頼ることができます。


In [ ]:
from genai_processors import switch

input_stream = streams.stream_content([
    content_api.ProcessorPart("a1", substream_name="a"),
    content_api.ProcessorPart("b1", substream_name="b"),
    content_api.ProcessorPart("a2", substream_name="a"),
    content_api.ProcessorPart("b2", substream_name="b"),
    content_api.ProcessorPart("b3", substream_name="b"),
])

m = (
    switch.Switch(content_api.get_substream_name)
    .case("a", another_text_processor)
    .case("b", simple_text_processor)
    .default(processor.passthrough())
)

print("\nSwitch Processor Output:")
async for part in m(input_stream):
  print(part)

`match` は並列ではなく、case で定義した条件に基づいて入力パートをルーティングするスイッチ型のプロセッサです。最初に一致した case の条件が、そのパートに対して使用されるプロセッサを決定します。一致する case がない場合は default が実行されます。この例では、パートをそのまま渡します。default を省略すると、一致する case がないときは何も返されません。

**ヒント**: スイッチ型プロセッサは、プロセッサのフロー内でパートに基づく分岐条件を定義するのに使えます。

## 7. 🤖 GenAI モデルをプロセッサとして使用する

このライブラリには、Google の生成 AI モデルと連携するための組み込みプロセッサが用意されています。上で API キーを設定していない場合は、このセクションはスキップしてください。


In [ ]:
from genai_processors.core import genai_model
from google.genai import types as genai_types

# Initialize the GenAI model processor
# Replace 'gemini-2.0-flash' with your desired model name
genai_processor = genai_model.GenaiModel(
    api_key=API_KEY,
    model_name="gemini-2.0-flash",
    generate_content_config=genai_types.GenerateContentConfig(temperature=0.7),
)

# Chain the GenAI processor with a processor to lowercase all inputs.
genai_pipeline = another_text_processor + genai_processor

input_prompt_genai = [
    "Explain the Concept of LARGE LANGUAGE MODELS",
    "in two sentences",
]
input_stream_genai = streams.stream_content(input_prompt_genai)

print("\nGenAI Pipeline Output:")
async for part in genai_pipeline(input_stream_genai):
  print(part.text)

GenAI Processors ライブラリには、任意の一方向ストリーミング Processor で First Token までの時間（TTFT: Time-To-First-Token）を記録する `TTFTSingleStream` プロセッサが用意されています。これは入力プロセッサをラップして元のロジックは保ったまま、呼び出しから最初の出力までの時間を記録します。この TTFT プロセッサは双方向モデル（LiveProcessor）には使用できません。


In [ ]:
from genai_processors import debug

# Chain the GenAI processor with a processor to lowercase all inputs.
genai_pipeline = (
    another_text_processor
    # Add a tag "GenAI Model" to which processor the TTFT applies to
    + debug.TTFTSingleStream("GenAI Model", genai_processor)
)

input_prompt_genai = [
    "Explain the Concept of LARGE LANGUAGE MODELS",
    "in two sentences",
]
input_stream_genai = streams.stream_content(input_prompt_genai)

print("\nGenAI Pipeline Output:")
async for part in genai_pipeline(input_stream_genai):
  print(part.text)

`GenAI Model TTFT=x.xx seconds` はプロセッサ出力の前に表示され、`status` サブストリームで返されます。

以下の例に示すように、プロセッサを使って Live API に容易に接続することもできます。


In [ ]:
from genai_processors.core import live_model
from google.genai import types as genai_types
from IPython.display import Audio, display
import numpy as np

LIVE_MODEL_NAME = "gemini-2.0-flash-live-001"

live_processor = live_model.LiveProcessor(
    api_key=API_KEY,
    model_name=LIVE_MODEL_NAME,
    realtime_config=genai_types.LiveConnectConfig(
        # Basic configuration for real-time text and audio interaction
        output_audio_transcription={},  # Enable transcription of audio output
        realtime_input_config=genai_types.RealtimeInputConfig(
            turn_coverage=(  # Model sees all real-time input in a turn
                "TURN_INCLUDES_ALL_INPUT"
            )
        ),
        response_modalities=["AUDIO"],  # Request audio output
    ),
)


@processor.processor_function
async def collect_audio(
    content: AsyncIterable[content_api.ProcessorPart],
) -> AsyncIterable[content_api.ProcessorPart]:
  """Yields a single Part containing all the audio from `content`."""
  audio_bytes = b""
  async for part in content:
    if content_api.is_audio(part.mimetype):
      audio_bytes += part.bytes
    elif content_api.is_text(part.mimetype):
      print(part)
  # This is yielded when the input stream is closed.
  yield content_api.ProcessorPart(
      audio_bytes,
      mimetype="audio/l16;rate=24000",
  )


# We only add text here, but this can contain audio, images, etc. This would
# typically come from a camera, microphone, or other input source.
input_stream = streams.stream_content(
    [
        content_api.ProcessorPart(
            "How are you today?", substream_name="realtime"
        )
    ],
    # This is needed for this example only: we wait here to give enough time
    # for the model to generate audio before we close the stream.
    with_delay_sec=7,
)
print("\nLive Processor Output:")
p = live_processor + collect_audio
async for part in p(input_stream):
  audio_track = Audio(
      data=np.frombuffer(part.bytes, dtype=np.int16),
      rate=24000,
      autoplay=True,
  )
  display(audio_track)

## 8. 🧩 PartProcessor の活用

各 `ProcessorPart` を個別に処理する操作には、`PartProcessor` クラスまたは `@processor.part_processor_function` デコレータを利用できます。`PartProcessor` は `to_processor()` メソッドで `Processor` にキャストできます。結果の Processor は、入力中のすべてのパートに対して並行に実行され、パートの順序は保持されます。`Processor` と連結する際、`PartProcessor` は暗黙的に `Processor` にキャストされます。そのため常に `to_processor()` が必要というわけではありません。ただし、`AsyncIterable[content_api.ProcessorPart]` に `PartProcessor` を適用したい場合は、このメソッドを実行する必要があります。

`PartProcessor` を定義する際には、このプロセッサが処理対象とする `Part` の種類を定義する `match` 関数を追加できます。任意ですが、指定することを推奨します。これは GenAI Processors ライブラリが asyncio タスクをどのようにスケジュールするかを最適化するために使用されます。

`match` 関数のシグネチャは次のとおりです:

```python
def match(part: content_api.ProcessorPart) -> bool:
  """Returns False if `part` is irrelevant for the processor, True otherwise."""
  ...
```

デフォルト実装は `True` を返し、すべてのパートがプロセッサの対象であるとみなします。関係のないパートに対して `True` を返しても構いません。その場合、そのパートは処理された後に無視されます。一方で、`False` を返す場合は正確であることが重要です。`match` が `False` を返したパートは一切処理されません。

`match` のデフォルト実装は `PartProcessor` クラスでオーバーライドできるほか、以下のように `@processor.part_processor_function` デコレータの追加パラメータとしてアドホックに指定することもできます。


In [ ]:
def match_text(part: content_api.ProcessorPart) -> bool:
  return content_api.is_text(part.mimetype)


@processor.part_processor_function(match_fn=match_text)
async def duplicate_part(
    part: content_api.ProcessorPart,
) -> AsyncIterable[content_api.ProcessorPart]:
  """Duplicates the input part."""
  yield part
  yield part


input_parts_duplicate = streams.stream_content(["A", "B"])

# To apply `duplicate_part` on the input *stream*, we need a Processor.
p = duplicate_part.to_processor()

print("\nPart Processor Output:")
async for part in p(input_parts_duplicate):
  print(part.text)

これは `A`, `A`, `B`, `B` を出力します。

このライブラリは、`Callable[[ProcessorPart], bool]` を受け取る `create_filter` メソッドを使って、`PartProcessor` としてフィルタを簡単に作成する方法も提供しています:

```python
# Creates a PartProcessor that only outputs the text parts. All other parts
# are dropped.
p = processor.create_filter(content_api.is_text)
```

### `to_processor()` に関する注意

PartProcessor は Processor インターフェースを実装していないため、ときに Processor と混同して次のような誤ったコードを書いてしまうことがあります:

```python
# p is a  PartProcessor defined somewhere else in the code.
p = part_processor

async def my_processor(
    content: AsyncIterable[ProcessorPart]
) -> AsyncIterable[ProcessorPart]:
  # This is an error as `p` expects a ProcessorPart and not an AsyncIterable.
  async for part in p(content):
    ...
```

これは、`to_processor()` メソッドを使って PartProcessor を Processor にキャストすることで簡単に修正できます。

```python
# p is a now a processor.
p = part_processor.to_processor()

async def my_processor(
    content: AsyncIterable[ProcessorPart]
) -> AsyncIterable[ProcessorPart]:
  # This is ok, `p` accepts AsyncIterables as input.
  async for part in p(content):
    ...
```

`to_processor` メソッドは Processor と PartProcessor のどちらにも適用できます。迷ったときは、Processor を AsyncIterable と併用できるようにするために、このメソッドを積極的に使ってください。

## 9. 🏎️ PartProcessor の並列実行

複数の PartProcessor インスタンスは `//` 演算子で並列に実行できます。


In [ ]:
@processor.part_processor_function
async def append_star(
    part: content_api.ProcessorPart,
) -> AsyncIterable[content_api.ProcessorPart]:
  """Appends a star to the text."""
  if content_api.is_text(part.mimetype):
    yield content_api.ProcessorPart(part.text + "*")


@processor.part_processor_function
async def append_hash(
    part: content_api.ProcessorPart,
) -> AsyncIterable[content_api.ProcessorPart]:
  """Appends a hash to the text."""
  if content_api.is_text(part.mimetype):
    yield content_api.ProcessorPart(part.text + "#")


parallel_processors = append_star // append_hash // processor.PASSTHROUGH_ALWAYS

input_parts_parallel = streams.stream_content([
    "Item_1",
    "Item_2",
    content_api.ProcessorPart(b"", mimetype="audio/l16;rate=24000"),
])

print("\nParallel Part Processors Output:")
async for part in parallel_processors.to_processor()(input_parts_parallel):
  print(part)

出力は `Item_1*`, `Item_1#`, `Item_2*`, `Item_2#`, `<audio part>` となります。

`//` 演算子は `PartProcessor` にのみ適用されます。すべての PartProcessor は入力パートに対して並行に実行され、その出力シーケンスは `//` 式で指定した順序で連結されます。この例では、式内で `append_star` が `append_hash` より前にあるため、星の追加がハッシュの追加より先に行われます。入力の順序も保持され、シーケンス内では `Item_1` が `Item_2` より前に現れます。

効率のため、`//` 式で複数の PartProcessor に入力パートを渡す際に入力はコピーされず、同じオブジェクトが渡されます。したがって、各 PartProcessor は入力 Part 引数のミュータブルな属性を変更しないようにしてください。

個々の PartProcessor のいずれからも出力が返されない場合、デフォルトでは全体の式からも何も返されません。`//` グループ内のいずれのプロセッサも何も返さなかったときに、入力パートをそのまま返す特別モードを有効にできます:

```python
parallel_processors = (
  append_star // append_hash // processor.PASSTHROUGH_FALLBACK
)
```

この `//` 演算子は、入力タイプに応じて事前処理を行うチャンクプロセッサを構成する際に便利です。典型的なパターンは次のとおりで、`xx_processor` が特定のパート型に対する前処理を定義します:

```python
p1 = processor.create_filter(content_api.is_image) + image_processor
p2 = processor.create_filter(content_api.is_audio) + audio_processor
total_processor = p1 // p2 // processor.PASSTHROUGH_FALLBACK
```

Processor と同様に、PartProcessor にもスイッチ文があります。これは PartProcessor で動作し、入力ストリームと出力ストリームのパート順序が同じになるように保証します。処理自体は並行して行われるため、実行において順序が重要にならないよう設計してください。


In [ ]:
from genai_processors import switch

input_stream = streams.stream_content([
    content_api.ProcessorPart("a1", substream_name="a"),
    content_api.ProcessorPart("b1", substream_name="b"),
    content_api.ProcessorPart("a2", substream_name="a"),
    content_api.ProcessorPart("b2", substream_name="b"),
    content_api.ProcessorPart("b3", substream_name="b"),
])

m = (
    switch.PartSwitch(content_api.get_substream_name)
    .case("a", append_star)
    .case("b", append_hash)
    .default(processor.passthrough())
)

print("\nPartSwitch Output:")
p = m.to_processor()
async for part in p(input_stream):
  print(part)

出力ストリームの順序は入力ストリームの順序と一致している点に注意してください。

`Switch` と `PartSwitch` の case 条件は同じです。両者の主な違いは、型（Processor か PartProcessor か）と、片方（PartProcessor）では順序が保持され、もう片方では保持されないという点です。Switch Processor は、同一の case 条件と同一の Processor を通るチャンクに対してのみ順序を保持します。

## 10. 🧱 異なるコンテンツタイプの扱い

`content_api` モジュールは、テキスト、画像、カスタム構造化データなど、`ProcessorPart` 内のさまざまなコンテンツタイプを扱うためのユーティリティを提供します。


In [ ]:
import io
from PIL import Image

# Create a simple black image
img = Image.new("RGB", (60, 30), color="black")
img_byte_arr = io.BytesIO()
img.save(img_byte_arr, format="PNG")
img_bytes = img_byte_arr.getvalue()

image_part = content_api.ProcessorPart(img_bytes, mimetype="image/png")
text_part = content_api.ProcessorPart("Some text")

# Accessing content
print("\nContent API Examples:")
print(f"Text part text: {text_part.text}")
print(f"Image part mimetype: {image_part.mimetype}")

# Using content_api.as_text to extract text from a list of parts
all_parts = [text_part, image_part, content_api.ProcessorPart(" more text")]
print(f"Combined text from parts: {content_api.as_text(all_parts)}")

## 11. ⏩ Streams と AsyncIterable の扱い

Processor は `ProcessorPart` の `AsyncIterable` ストリームを処理します。`streams` モジュールには、これらのストリームを扱うのに役立つ関数が用意されています。

### Iterable を AsyncIterable に変換する

`streams.stream_content` 関数は、（リストなどの）標準的な Python の iterable を `AsyncIterable` に変換します。これは Processor で処理するために必要です。


In [ ]:
import asyncio
from genai_processors import content_api, streams

iterable_data = ["Part 1", "Part 2", "Part 3"]
async_stream = streams.stream_content(iterable_data)

print("\nProcessing stream:")
async for part in async_stream:
  print(f"Received: {part}")

これは主に、リストから手軽に `AsyncIterable` を作成するためのテストで使用されます。`stream_content` に `with_delay_sec` 引数を渡すと、すべての要素が即座に生成されないようにできます。

### ストリームをリストに集約する

`streams.gather_stream` 関数は、`AsyncIterable` からすべての要素を収集して Python のリストにまとめます。有限のストリームの出力をすべて消費したい場合に便利です。


In [ ]:
import asyncio
from genai_processors import content_api, streams

async_stream = streams.stream_content(
    [content_api.ProcessorPart("A"), content_api.ProcessorPart("B")]
)
gathered_list = await streams.gather_stream(async_stream)

print("\nGathered list from stream:")
print(gathered_list)

### ストリームの分割と結合

`streams` モジュールには、単一のストリームを複数の同一ストリームに分割する機能（`streams.split`）や、複数のストリームを1つに結合する機能（`streams.merge` と `streams.concat`）があります。これは、パイプラインの異なる部分が同じ入力で動作したり、異なるソースからの結果を統合したりする必要がある複雑な処理グラフを構築するのに有用です。


In [ ]:
import asyncio
from genai_processors import content_api, processor, streams


@processor.processor_function
async def append_a(
    content: AsyncIterable[content_api.ProcessorPart],
) -> AsyncIterable[content_api.ProcessorPart]:
  async for part in content:
    yield content_api.ProcessorPart(part.text + "A")


@processor.processor_function
async def append_b(
    content: AsyncIterable[content_api.ProcessorPart],
) -> AsyncIterable[content_api.ProcessorPart]:
  async for part in content:
    yield content_api.ProcessorPart(part.text + "B")


initial_stream = streams.stream_content(
    ["Start", "Finish"],
    # We add a delay after yielding each item. This lets the "Start" items be
    # yielded first.
    with_delay_sec=0.001,
)

# Split the stream into two
stream1, stream2 = streams.split(initial_stream, n=2)

# Process each stream independently
processed_stream1 = append_a(stream1)
processed_stream2 = append_b(stream2)

# Merge the processed streams
merged_stream = streams.merge([processed_stream1, processed_stream2])

print("\nSplit and Merge Example Output:")
async for part in merged_stream:
  print(part.text)

この例では、最初のストリームを分割し、それぞれの枝で処理（片方に"A"、もう片方に"B"を付加）した後、結果をマージしています。マージ後の出力順序はタスクのスケジューリングによって変わることがあります。ここでは `with_delay_sec` を設定して、まず `start` の要素がすべて先に生成されるようにしています。これを外すと、スケジューリングはおそらく異なるものになります。

`merge` と `queues` を用いることで、次のようにストリーム内にループを作ることもできます:


In [ ]:
input_stream = streams.stream_content(
    [content_api.ProcessorPart("Hello"), content_api.ProcessorPart("World")],
    # Adds a 0.1 second delay after streaming each part. This is needed in this
    # example to insert the content of the input_queue into the stream before it
    # is closed.
    with_delay_sec=0.1,
)
input_queue = asyncio.Queue()
stream_loop = streams.merge(
    [input_stream, streams.dequeue(input_queue)],
    stop_on_first=True,
)


async def inject_new_part():
  async for part in append_a(stream_loop):
    print(part.text)
    # Wait for 0.09 seconds to inject a new part before the next part is
    # streamed.
    await asyncio.sleep(0.09)
    # Inject a "new_part" Part in the stream_loop.
    await input_queue.put(content_api.ProcessorPart("new_part"))


# This will output: HelloA, new_partA, WorldA
asyncio.run(inject_new_part())

`new_part` は `input_queue` を通じて `stream_loop` に注入されたため、`append_a` プロセッサで処理されている点に注意してください。ここでは `input_stream` が終わったときに停止させるため、`stop_on_first` を `True` に設定する必要があります。そうしないと、`live_queue` が終了しないため、`new_part` によってループが無限に自己充填されてしまいます。

このイディオムを用いると、プロセッサの出力を再びプロセッサに再注入して処理できる、リアルタイムエージェントでよくある複雑なパイプラインを構築できます。


### Processor ソースからストリームを作成する

入力ストリームがマイク、カメラ、ネットワーク接続などの外部ソースに由来する場合、`@processor.source` デコレータを使うと、`ProcessorPart` のストリームを生成する「ソース」を簡単に定義できます。これは、既存のストリームを変換するのではなく、`ProcessorPart` のストリームを「開始」する特別な種類の `Processor` と考えてください。

ソースは、生成したい `ProcessorPart` を `yield` する `async` ジェネレーター関数を記述して定義します。例えば、以下はターミナルから入力を読み込むソースです:


In [ ]:
@processor.source()
async def TerminalInput(
    prompt: str,
) -> AsyncIterable[content_api.ProcessorPartTypes]:
  while True:
    input_text = await asyncio.to_thread(input, prompt)
    if input_text == 'q':
      break
    yield input_text


async for part in TerminalInput('>'):
  print(part)

`@processor.source` デコレータは、あなたのジェネレーターを自動的に本格的な `Processor` に変換します。

これは次のことを意味します:

1.  **連結できる:** `+` 演算子を使って、複数のソース（あるいはソースと通常のプロセッサ）を組み合わせることができます。あるソースからのパートは、次のプロセッサが生成するパートとマージされます。

    ```python
    # Imagine audio_io.AudioIn and live_model.LiveModel are other processors
    # that produce or process data.
    p = TerminalInput('>') + audio_io.AudioIn(...) + live_model.LiveModel(...)
    # Here, TerminalInput starts the stream, which is then combined with
    # audio input, and finally fed into a live model.
    # endless_stream() is an empty stream that never ends.
    # It is often used when a source initiates the stream.
    async for part in p(streams.endless_stream()):
        # Process the combined output
        pass
    ```

    ソースを連結する場合、その入力ストリーム（例: `streams.endless_stream()`）は主にソースの動作を開始するために使われます。ソースはこのストリームに自分が生成したパートを追加し、その結果、チェーン内の後続プロセッサは初期ストリームとソースの両方からデータを受け取れるようになります。

2.  **入力ストリームとして使える:** ソース自体が `ProcessorPart` の `AsyncIterable` として振る舞うため、任意の `Processor` の入力として直接渡すことができます。

    ```python
    # Here, TerminalInput generates parts, and live_model processes them.
    async for part in live_model.LiveModel(...)(TerminalInput('>')):
        # Process parts from the live model
        pass
    ```


## 12. ➡️ 次のステップ

このチュートリアルでは、プロセッサの作成・適用・連結の基本に加え、さまざまなコンテンツタイプや GenAI モデルの扱いについて説明しました。

新しい `Processor` の開発をさらに深く学ぶには、[create_your_own_processor](https://colab.research.google.com/github/google-gemini/genai-processors/blob/main/notebooks/create_your_own_processor.ipynb) ノートブックに進んでください。
